In [ ]:
# CELL 1 — ENVIRONMENT AND PROJECT SETUP

%pip install -q geopandas pyogrio shapely pyproj fiona requests pyyaml

from pathlib import Path
import requests
import yaml
import json
import geopandas as gpd
import pandas as pd
import shapely
import pyproj
import sys

# Main project directory
PROJECT = Path("/kaggle/working/recreation_boundary_pipeline")

# Folder structure
DIRS = {
    "raw": PROJECT / "raw",
    "processed": PROJECT / "processed",
    "outputs": PROJECT / "outputs",
    "logs": PROJECT / "logs",
    "metadata": PROJECT / "metadata",
}

for path in DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

# City folders for original source files
CITIES = [
    "pune",
    "mumbai",
    "hyderabad",
    "chennai",
    "kolkata",
    "ahmedabad",
    "jaipur",
    "lucknow",
    "bhubaneswar",
    "indore",
]

for city in CITIES:
    (DIRS["raw"] / city).mkdir(parents=True, exist_ok=True)

# Retrieve the repository's actual dataset catalogue
CATALOG_URL = (
    "https://raw.githubusercontent.com/"
    "yashveeeeeeer/india-geodata/main/docs/_data/catalog.yml"
)

response = requests.get(CATALOG_URL, timeout=60)
response.raise_for_status()

catalog_path = DIRS["metadata"] / "india_geodata_catalog.yml"
catalog_path.write_bytes(response.content)

with open(catalog_path, "r", encoding="utf-8") as f:
    CATALOG = yaml.safe_load(f)

print("Environment ready.")
print("Project directory:", PROJECT)
print("Catalogue saved:", catalog_path)
print("GeoPandas:", gpd.__version__)
print("Shapely:", shapely.__version__)
print("PyProj:", pyproj.__version__)
print("Python:", sys.version.split()[0])

In [ ]:
# CELL 2 — INSPECT THE INDIA GEODATA CATALOGUE

from pprint import pprint

print("Catalogue type:", type(CATALOG).__name__)

if isinstance(CATALOG, dict):
    print("\nTop-level keys:")
    pprint(list(CATALOG.keys()))

elif isinstance(CATALOG, list):
    print("\nNumber of top-level entries:", len(CATALOG))
    print("\nFirst three entries:")
    pprint(CATALOG[:3])

else:
    print("\nCatalogue content:")
    pprint(CATALOG)

print("\n" + "=" * 60)
print("CITY MATCHES")
print("=" * 60)

ALIASES = {
    "pune": ["pune"],
    "mumbai": ["mumbai"],
    "hyderabad": ["hyderabad"],
    "chennai": ["chennai"],
    "kolkata": ["kolkata"],
    "ahmedabad": ["ahmedabad", "amdavad"],
    "jaipur": ["jaipur"],
    "lucknow": ["lucknow"],
    "bhubaneswar": ["bhubaneswar"],
    "indore": ["indore"],
}

def find_city_entries(obj, path="root"):
    if isinstance(obj, dict):
        for key, value in obj.items():
            new_path = f"{path}.{key}"
            find_city_entries(value, new_path)

    elif isinstance(obj, list):
        for i, value in enumerate(obj):
            find_city_entries(value, f"{path}[{i}]")

    elif isinstance(obj, str):
        value_lower = obj.lower()
        for city, names in ALIASES.items():
            if any(name in value_lower for name in names):
                print(f"\n[{city.upper()}]")
                print("Path:", path)
                print("Value:", obj[:1000])

find_city_entries(CATALOG)

In [ ]:
# CELL 3 — INSPECT MUNICIPAL BOUNDARY SOURCES

for entry in CATALOG:
    if "municipal-boundaries" in str(entry.get("url", "")).lower():
        print("\n" + "=" * 80)
        print("TITLE:", entry.get("title"))
        print("DESCRIPTION:", entry.get("description"))
        print("URL:", entry.get("url"))
        print("LICENSE:", entry.get("license_id"))
        print("-" * 80)

        for file in entry.get("files", []):
            print(
                file.get("name"),
                "|",
                file.get("format"),
                "|",
                file.get("download_url")
            )

In [ ]:
# CELL 4 — INSPECT CITY BOUNDARY SOURCE DOCUMENTATION

import requests

TARGET_CITIES = [
    "ahmedabad", "bhubaneswar", "chennai", "hyderabad",
    "jaipur", "kolkata", "lucknow", "mumbai", "pune"
]

boundary_entry = next(
    entry for entry in CATALOG
    if entry.get("url") == "data/urban/municipal-boundaries"
)

for city in TARGET_CITIES:
    print("\n" + "=" * 80)
    print(city.upper())
    print("=" * 80)

    readme_files = [
        file for file in boundary_entry.get("files", [])
        if file.get("name", "").lower().startswith(city + "/")
        and file.get("name", "").lower().endswith("readme.md")
    ]

    if not readme_files:
        print("No README listed.")
        continue

    for file in readme_files:
        print("\nFile:", file["name"])
        try:
            r = requests.get(file["download_url"], timeout=60)
            r.raise_for_status()
            print(r.text[:10000])
        except Exception as e:
            print("Could not retrieve README:", repr(e))

print("\n" + "=" * 80)
print("INDORE CATALOGUE SEARCH")
print("=" * 80)

for entry in CATALOG:
    matches = [
        f for f in entry.get("files", [])
        if "indore" in f.get("name", "").lower()
    ]
    if matches:
        print("Dataset:", entry.get("title"))
        for f in matches:
            print(f.get("name"), "|", f.get("download_url"))

In [ ]:
# CELL 5 — DOWNLOAD AND INSPECT EIGHT CANDIDATE BOUNDARIES

import requests
import geopandas as gpd

SELECTED_FILES = {
    "pune": "pune/pune-electoral-wards_2022.geojson",
    "mumbai": "mumbai/BMC_Wards.geojson",
    "hyderabad": "hyderabad/ghmc-area.geojson",
    "chennai": "chennai/Wards.geojson",
    "kolkata": "kolkata/kolkata.geojson",
    "jaipur": "jaipur/Jaipur_Boundary.geojson",
    "lucknow": "lucknow/Lucknow_city_boundary.geojson",
    "bhubaneswar": "bhubaneswar/BMC Boundary.GeoJSON",
}

boundary_entry = next(
    entry for entry in CATALOG
    if entry.get("url") == "data/urban/municipal-boundaries"
)

catalogue_files = {
    f["name"]: f
    for f in boundary_entry.get("files", [])
}

downloaded_paths = {}
inspection_results = []

for city, filename in SELECTED_FILES.items():
    print("\n" + "=" * 70)
    print(city.upper())

    if filename not in catalogue_files:
        print("ERROR: Exact file not found in catalogue:", filename)
        continue

    file_info = catalogue_files[filename]
    url = file_info["download_url"]
    destination = DIRS["raw"] / city / Path(filename).name

    try:
        response = requests.get(url, timeout=120)
        response.raise_for_status()

        destination.write_bytes(response.content)
        gdf = gpd.read_file(destination)

        downloaded_paths[city] = destination

        result = {
            "city": city,
            "file": filename,
            "features": len(gdf),
            "crs": str(gdf.crs),
            "geometry_types": sorted(gdf.geom_type.dropna().unique().tolist()),
            "valid_geometries": int(gdf.geometry.is_valid.sum()),
            "total_geometries": int(gdf.geometry.notna().sum()),
            "empty_geometries": int(gdf.geometry.is_empty.sum()),
        }
        inspection_results.append(result)

        print("Downloaded:", destination)
        print("Features:", result["features"])
        print("CRS:", result["crs"])
        print("Geometry types:", result["geometry_types"])
        print(
            "Valid geometries:",
            f'{result["valid_geometries"]}/{result["total_geometries"]}'
        )
        print("Empty geometries:", result["empty_geometries"])
        print("Columns:", list(gdf.columns))

    except Exception as e:
        print("ERROR:", repr(e))

inspection_df = pd.DataFrame(inspection_results)

print("\n" + "=" * 70)
print("DOWNLOAD SUMMARY")
display(inspection_df)

In [ ]:
from pathlib import Path

KML_FILES = list(Path("/kaggle/input").rglob("*.kml"))

print(f"Found {len(KML_FILES)} KML files:\n")

for i, path in enumerate(KML_FILES, 1):
    print(f"{i}. {path}")

In [ ]:
import fiona
import pyogrio

print("Fiona KML support:")
print({k: v for k, v in fiona.supported_drivers.items()
       if "KML" in k.upper()})

print("\nGDAL drivers through Pyogrio:")
drivers = pyogrio.list_drivers()
print({k: v for k, v in drivers.items()
       if "KML" in k.upper()})

In [ ]:
import shutil
import geopandas as gpd

KML_PATHS = {
    "ahmedabad": Path(
        "/kaggle/input/datasets/vedantsaraswate/indore-ahemdabad-wards/6a6d025b-2b02-4b0e-8bb8-74186eacf071.kml"
    ),
    "indore": Path(
        "/kaggle/input/datasets/vedantsaraswate/indore-ahemdabad-wards/9b8f3f8a-3480-42d5-bc1c-09c9d69c4717.kml"
    ),
}

for city, source_path in KML_PATHS.items():
    # Preserve original KML
    raw_path = DIRS["raw"] / city / f"{city}_wards.kml"
    shutil.copy2(source_path, raw_path)

    # Read using Pyogrio
    gdf = gpd.read_file(
        source_path,
        engine="pyogrio",
        driver="KML"
    )

    if gdf.crs is None:
        raise ValueError(f"{city}: CRS is missing.")

    gdf = gdf.to_crs("EPSG:4326")

    print(f"\n{city.upper()}")
    print("Features:", len(gdf))
    print("Columns:", gdf.columns.tolist())
    print("CRS:", gdf.crs)
    print("Geometry types:", gdf.geom_type.value_counts().to_dict())
    print("Null geometries:", gdf.geometry.isna().sum())
    print("Empty geometries:", gdf.geometry.is_empty.sum())
    print("Valid geometries:", gdf.geometry.is_valid.sum())

    output_path = DIRS["processed"] / f"{city}_wards.geojson"
    gdf.to_file(output_path, driver="GeoJSON", engine="pyogrio")

    print("Saved:", output_path)

In [ ]:
BOUNDARY_INPUTS = {
    "pune": DIRS["raw"] / "pune" / "pune-electoral-wards_2022.geojson",
    "mumbai": DIRS["raw"] / "mumbai" / "BMC_Wards.geojson",
    "hyderabad": DIRS["raw"] / "hyderabad" / "ghmc-area.geojson",
    "chennai": DIRS["raw"] / "chennai" / "Wards.geojson",
    "kolkata": DIRS["raw"] / "kolkata" / "kolkata.geojson",
    "ahmedabad": DIRS["processed"] / "ahmedabad_wards.geojson",
    "jaipur": DIRS["raw"] / "jaipur" / "Jaipur_Boundary.geojson",
    "lucknow": DIRS["raw"] / "lucknow" / "Lucknow_city_boundary.geojson",
    "bhubaneswar": DIRS["raw"] / "bhubaneswar" / "BMC Boundary.GeoJSON",
    "indore": DIRS["processed"] / "indore_wards.geojson",
}

BOUNDARIES = {}

for city, path in BOUNDARY_INPUTS.items():
    if not path.exists():
        print(f"\n{city.upper()}: FILE NOT FOUND — {path}")
        continue

    gdf = gpd.read_file(path, engine="pyogrio")
    BOUNDARIES[city] = gdf

    print(f"\n{'='*45}")
    print(city.upper())
    print("Features:", len(gdf))
    print("CRS:", gdf.crs)
    print("Geometry types:", gdf.geom_type.value_counts().to_dict())
    print("Null:", gdf.geometry.isna().sum())
    print("Empty:", gdf.geometry.is_empty.sum())
    print("Valid:", gdf.geometry.is_valid.sum())
    print("Invalid:", (~gdf.geometry.is_valid).sum())

    invalid = gdf.loc[~gdf.geometry.is_valid]
    for idx, row in invalid.iterrows():
        print(
            "Invalid feature:",
            idx,
            "| Reason:",
            shapely.is_valid_reason(row.geometry)
        )

print("\nSuccessfully loaded:", len(BOUNDARIES), "/ 10 cities")

In [ ]:
import shapely
import geopandas as gpd

kolkata_original = BOUNDARIES["kolkata"].copy()
kolkata_repaired = kolkata_original.copy()

# Identify invalid features
invalid_mask = ~kolkata_repaired.geometry.is_valid
print("Invalid features:", invalid_mask.sum())

# Repair only invalid geometries
for idx in kolkata_repaired.index[invalid_mask]:
    original = kolkata_repaired.at[idx, "geometry"]
    repaired = shapely.make_valid(original)

    print(f"\nFeature {idx}")
    print("Original type:", original.geom_type)
    print("Repaired type:", repaired.geom_type)
    print("Repaired valid:", repaired.is_valid)
    print("Original area (degrees²):", original.area)
    print("Repaired area (degrees²):", repaired.area)

    kolkata_repaired.at[idx, "geometry"] = repaired

# Check complete dataset
print("\nFinal validity:")
print("Valid:", kolkata_repaired.geometry.is_valid.sum())
print("Invalid:", (~kolkata_repaired.geometry.is_valid).sum())
print("Geometry types:", kolkata_repaired.geom_type.value_counts().to_dict())

# Compare total area in an equal-area CRS
original_area = kolkata_original.to_crs("EPSG:6933").geometry.area.sum()
repaired_area = kolkata_repaired.to_crs("EPSG:6933").geometry.area.sum()

area_change = repaired_area - original_area
percentage_change = (area_change / original_area) * 100

print("\nArea comparison (km²):")
print("Original:", original_area / 1e6)
print("Repaired:", repaired_area / 1e6)
print("Difference:", area_change / 1e6)
print("Percentage change:", percentage_change)

# Save as a separate candidate; original remains untouched
repair_path = DIRS["processed"] / "kolkata_wards_repaired_candidate.geojson"
kolkata_repaired.to_file(repair_path, driver="GeoJSON", engine="pyogrio")

print("\nSaved candidate:", repair_path)

In [ ]:
STANDARDISED = {}

for city, gdf in BOUNDARIES.items():

    if city == "kolkata":
        gdf = gpd.read_file(
            DIRS["processed"] / "kolkata_wards_repaired_candidate.geojson",
            engine="pyogrio"
        )

    gdf = gdf.copy()

    if gdf.crs is None:
        raise ValueError(f"{city}: Missing CRS")

    gdf = gdf.to_crs("EPSG:4326")

    if gdf.geometry.isna().any() or gdf.geometry.is_empty.any():
        raise ValueError(f"{city}: Null or empty geometries found")

    invalid_count = (~gdf.geometry.is_valid).sum()
    if invalid_count:
        raise ValueError(f"{city}: {invalid_count} invalid geometries remain")

    output_path = DIRS["processed"] / f"{city}_wards_standardised.geojson"
    gdf.to_file(output_path, driver="GeoJSON", engine="pyogrio")

    STANDARDISED[city] = gdf

    print(
        f"{city.title():15} | "
        f"{len(gdf):3} features | "
        f"{gdf.geom_type.nunique()} geometry type(s) | "
        f"Valid"
    )

print("\nStandardised:", len(STANDARDISED), "/ 10 cities")

In [ ]:
import pandas as pd
import geopandas as gpd

city_boundaries = []
area_records = []

for city, gdf in STANDARDISED.items():

    # Geometric union of all features in the city
    dissolved_geometry = gdf.geometry.union_all()

    if dissolved_geometry.is_empty or not dissolved_geometry.is_valid:
        raise ValueError(f"{city}: Dissolved geometry is invalid.")

    # Create one-feature GeoDataFrame
    boundary = gpd.GeoDataFrame(
        {
            "city": [city.title()],
            "source_features": [len(gdf)]
        },
        geometry=[dissolved_geometry],
        crs="EPSG:4326"
    )

    # Calculate area in an equal-area projected CRS
    area_km2 = boundary.to_crs("EPSG:6933").geometry.area.iloc[0] / 1e6

    boundary["area_km2"] = area_km2

    city_boundaries.append(boundary)

    area_records.append({
        "city": city.title(),
        "source_features": len(gdf),
        "geometry_type": dissolved_geometry.geom_type,
        "area_km2": area_km2,
        "valid": dissolved_geometry.is_valid
    })

    print(
        f"{city.title():15} | "
        f"{dissolved_geometry.geom_type:12} | "
        f"{area_km2:10.3f} km² | Valid"
    )

# Combine all ten municipal boundaries
FINAL_BOUNDARIES = gpd.GeoDataFrame(
    pd.concat(city_boundaries, ignore_index=True),
    geometry="geometry",
    crs="EPSG:4326"
)

# Save final boundary dataset
final_path = DIRS["outputs"] / "final_10_city_boundaries.geojson"
FINAL_BOUNDARIES.to_file(
    final_path,
    driver="GeoJSON",
    engine="pyogrio"
)

# Save area summary
AREA_SUMMARY = pd.DataFrame(area_records)
area_path = DIRS["outputs"] / "city_boundary_areas.csv"
AREA_SUMMARY.to_csv(area_path, index=False)

print("\nFinal boundary count:", len(FINAL_BOUNDARIES))
print("All valid:", FINAL_BOUNDARIES.geometry.is_valid.all())
print("Saved:", final_path)
print("Area summary:", area_path)

In [ ]:
%pip install -q osmnx

import osmnx as ox
import geopandas as gpd
import pandas as pd
from pathlib import Path

# Load final municipal boundaries
BOUNDARY_FILE = DIRS["outputs"] / "final_10_city_boundaries.geojson"
FINAL_BOUNDARIES = gpd.read_file(BOUNDARY_FILE, engine="pyogrio")

# All agreed OSM tags
RECREATION_TAGS = {
    "leisure": [
        "park", "garden", "nature_reserve", "dog_park",
        "playground", "water_park", "trampoline_park",
        "miniature_golf", "amusement_arcade", "escape_game",
        "sports_centre", "sports_hall", "stadium", "pitch",
        "track", "fitness_centre", "fitness_station",
        "swimming_pool", "ice_rink", "golf_course",
        "bowling_alley", "horse_riding",
        "picnic_table", "fishing", "marina", "slipway",
        "bathing_place", "swimming_area", "bird_hide",
        "wildlife_hide"
    ],
    "amenity": [
        "library", "arts_centre", "community_centre",
        "theatre", "planetarium", "exhibition_centre",
        "cinema", "music_venue", "events_venue", "nightclub"
    ],
    "tourism": [
        "museum", "gallery", "picnic_site",
        "viewpoint", "theme_park", "zoo"
    ],
    "landuse": [
        "recreation_ground", "village_green"
    ],
    "natural": [
        "beach"
    ]
}

# OSMnx configuration
ox.settings.use_cache = True
ox.settings.log_console = True
ox.settings.timeout = 180

print("OSMnx version:", ox.__version__)
print("Boundary cities:", FINAL_BOUNDARIES["city"].tolist())
print("Tag groups:", len(RECREATION_TAGS))
print("Setup complete.")

In [ ]:
print("Current Overpass endpoint:")
print(ox.settings.overpass_url)

print("\nTimeout:")
print(ox.settings.timeout)

print("\nCache enabled:")
print(ox.settings.use_cache)

print("\nAvailable OSMnx settings:")
print([x for x in dir(ox.settings) if "overpass" in x.lower()])

In [ ]:
import requests

TEST_URL = "https://overpass.private.coffee/api/interpreter"

try:
    response = requests.get(
        TEST_URL,
        params={"data": '[out:json][timeout:10];node(18.5204,73.8567,18.5205,73.8568);out;'},
        timeout=30
    )

    print("HTTP status:", response.status_code)
    print("Content type:", response.headers.get("Content-Type"))
    print("Response preview:", response.text[:300])

except requests.RequestException as e:
    print("Connection failed:", repr(e))

In [ ]:
import requests

TEST_URL = "https://maps.mail.ru/osm/tools/overpass/api/interpreter"

response = requests.get(
    TEST_URL,
    params={
        "data": '[out:json][timeout:10];node(18.5204,73.8567,18.5205,73.8568);out;'
    },
    headers={"User-Agent": PROJECT_UA},
    timeout=45
)

print("HTTP status:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print("Response preview:", response.text[:500])

In [ ]:
from pathlib import Path

OSM_FILES = list(Path("/kaggle/input").rglob("*.geojson.xz"))

print("Found:", len(OSM_FILES), "compressed GeoJSON files")

for path in OSM_FILES:
    print(path)

In [ ]:
import lzma
import json

OSM_PATH = Path(
    "/kaggle/input/datasets/vedantsaraswate/punefacilitiesdata/"
    "planet_73.7_18.417_51b1a268.osm.geojson.xz"
)

print("File size:", round(OSM_PATH.stat().st_size / (1024**2), 2), "MB")

with lzma.open(OSM_PATH, "rt", encoding="utf-8") as f:
    sample = f.read(6000)

print("\nFile opening:")
print(sample[:6000])

In [ ]:
%pip install -q ijson

import ijson
from collections import Counter

# Exact approved tags, grouped by category
TAG_CATEGORY_MAP = {
    "Green spaces": {
        "leisure": [
            "park", "garden", "nature_reserve",
            "dog_park"
        ],
        "landuse": [
            "recreation_ground", "village_green"
        ]
    },
    "Amusement and Play": {
        "leisure": [
            "playground", "water_park", "trampoline_park",
            "miniature_golf", "amusement_arcade", "escape_game"
        ]
    },
    "Sports": {
        "leisure": [
            "sports_centre", "sports_hall", "stadium", "pitch",
            "track", "fitness_centre", "fitness_station",
            "swimming_pool", "ice_rink", "golf_course",
            "bowling_alley", "horse_riding"
        ]
    },
    "Culture": {
        "amenity": [
            "library", "arts_centre", "community_centre",
            "theatre", "planetarium", "exhibition_centre"
        ],
        "tourism": ["museum", "gallery"]
    },
    "Entertainment": {
        "amenity": [
            "cinema", "music_venue", "events_venue", "nightclub"
        ],
        "tourism": ["theme_park", "zoo"]
    },
    "Outdoor activities": {
        "natural": ["beach"],
        "tourism": ["picnic_site", "viewpoint"],
        "leisure": [
            "picnic_table", "fishing", "marina", "slipway",
            "bathing_place", "swimming_area",
            "bird_hide", "wildlife_hide"
        ]
    }
}

# Reverse lookup: (OSM key, OSM value) -> category
TAG_LOOKUP = {
    (key, value): category
    for category, groups in TAG_CATEGORY_MAP.items()
    for key, values in groups.items()
    for value in values
}

tag_counts = Counter()
category_counts = Counter()
sample_records = []
property_keys = set()
total_features = 0
matched_features = 0

with lzma.open(OSM_PATH, "rb") as f:
    for feature in ijson.items(f, "features.item"):
        total_features += 1
        props = feature.get("properties") or {}
        property_keys.update(props.keys())

        matches = [
            (key, value, category)
            for (key, value), category in TAG_LOOKUP.items()
            if props.get(key) == value
        ]

        if not matches:
            continue

        matched_features += 1

        for key, value, category in matches:
            tag_counts[f"{key}={value}"] += 1
            category_counts[category] += 1

        if len(sample_records) < 15:
            sample_records.append({
                "geometry_type": feature.get("geometry", {}).get("type"),
                "matched_tags": matches,
                "properties": props
            })

print("Total features scanned:", total_features)
print("Matching features:", matched_features)

print("\nCounts by category:")
for category, count in category_counts.items():
    print(f"{category}: {count}")

print("\nMost common matching tags:")
for tag, count in tag_counts.most_common(25):
    print(f"{tag}: {count}")

print("\nAvailable OSM ID fields:")
print([k for k in sorted(property_keys) if "id" in k.lower()])

print("\nSample matching records:")
for record in sample_records:
    print({
        "geometry_type": record["geometry_type"],
        "matched_tags": record["matched_tags"],
        "properties": record["properties"]
    })

In [ ]:
from collections import Counter

geometry_counts = Counter()
missing_ids = 0
overlapping_categories = Counter()
access_counts = Counter()
sample_ids = []

with lzma.open(OSM_PATH, "rb") as f:
    for feature in ijson.items(f, "features.item"):
        props = feature.get("properties") or {}
        geom = feature.get("geometry") or {}

        matches = {
            category
            for (key, value), category in TAG_LOOKUP.items()
            if props.get(key) == value
        }

        if not matches:
            continue

        geometry_counts[geom.get("type")] += 1

        if feature.get("id") is None:
            missing_ids += 1
        elif len(sample_ids) < 10:
            sample_ids.append(feature.get("id"))

        if len(matches) > 1:
            overlapping_categories[tuple(sorted(matches))] += 1

        for key in ["access", "access:conditional", "fee", "operator:type"]:
            if key in props:
                access_counts[f"{key}={props[key]}"] += 1

print("Geometry types:", dict(geometry_counts))
print("Missing top-level IDs:", missing_ids)
print("Sample top-level IDs:", sample_ids)

print("\nFeatures matching multiple categories:")
for categories, count in overlapping_categories.items():
    print(categories, count)

print("\nAccess-related tags:")
for tag, count in access_counts.most_common(30):
    print(tag, count)

In [ ]:
from collections import Counter
import lzma
import ijson

overlap_records = []
access_counts = Counter()
access_examples = []

restricted_access = {
    "private", "members", "residents", "permit",
    "students", "children", "no", "discouraged"
}

conditional_access = {"customers"}

with lzma.open(OSM_PATH, "rb") as f:
    for feature in ijson.items(f, "features.item"):
        props = feature.get("properties") or {}

        matches = {
            category
            for (key, value), category in TAG_LOOKUP.items()
            if props.get(key) == value
        }

        if len(matches) > 1:
            overlap_records.append({
                "name": props.get("name"),
                "categories": sorted(matches),
                "facility_tags": {
                    k: v for k, v in props.items()
                    if k in ["leisure", "amenity", "tourism",
                             "landuse", "natural"]
                }
            })

        access = props.get("access")

        if access in restricted_access or access in conditional_access:
            access_counts[access] += 1

            if len(access_examples) < 5:
                access_examples.append({
                    "name": props.get("name"),
                    "access": access,
                    "facility_tags": {
                        k: v for k, v in props.items()
                        if k in ["leisure", "amenity", "tourism",
                                 "landuse", "natural"]
                    }
                })

print("CATEGORY OVERLAPS:", len(overlap_records))
for record in overlap_records:
    print(record)

print("\nACCESS COUNTS:")
for access, count in access_counts.most_common():
    print(f"{access}: {count}")

print("\nMAXIMUM FIVE ACCESS EXAMPLES:")
for example in access_examples:
    print(example)

In [ ]:
from collections import Counter
import lzma
import ijson

private_counts = Counter()
private_examples = []

with lzma.open(OSM_PATH, "rb") as f:
    for feature in ijson.items(f, "features.item"):
        props = feature.get("properties") or {}

        if props.get("access") != "private":
            continue

        matches = {
            category
            for (key, value), category in TAG_LOOKUP.items()
            if props.get(key) == value
        }

        if not matches:
            continue

        for category in matches:
            private_counts[category] += 1

        if len(private_examples) < 12:
            private_examples.append({
                "name": props.get("name"),
                "categories": sorted(matches),
                "tags": {
                    k: v for k, v in props.items()
                    if k in ["access", "leisure", "amenity",
                             "tourism", "landuse", "natural",
                             "fee", "operator"]
                }
            })

print("PRIVATE RECREATION FEATURES BY CATEGORY:")
for category, count in private_counts.most_common():
    print(category, count)

print("\nSAMPLE FEATURES (MAX 12):")
for item in private_examples:
    print(item)

In [ ]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

osm_files = sorted(
    p for p in INPUT_ROOT.rglob("*.xz")
    if "osm.geojson" in p.name
)

print("OSM extracts found:", len(osm_files))

for p in osm_files:
    print(p.name, "|", round(p.stat().st_size / 1024**2, 2), "MB")

In [ ]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

FILE_PREFIXES = {
    "Ahmedabad": "planet_71.84_22.326",
    "Mumbai": "planet_72.783_18.865",
    "Pune": "planet_73.7_18.417",
    "Jaipur": "planet_74.914_26.836",
    "Indore": "planet_75.809_22.668",
    "Hyderabad": "planet_78.384_17.385",
    "Chennai": "planet_80.134_13.045",
    "Lucknow": "planet_80.56_27.011",
    "Bhubaneswar": "planet_85.781_20.209",
    "Kolkata": "planet_88.234_22.549"
}

OSM_FILES = {}

for city, prefix in FILE_PREFIXES.items():
    matches = sorted(INPUT_ROOT.rglob(f"{prefix}*.osm.geojson.xz"))

    if not matches:
        raise FileNotFoundError(f"Missing extract: {city}")

    OSM_FILES[city] = matches[0]

    if len(matches) > 1:
        print(f"{city}: {len(matches)} copies found; using one.")

print("\nFINAL CITY MAPPING")
for city, path in OSM_FILES.items():
    print(f"{city}: {path}")

print("\nTotal mapped cities:", len(OSM_FILES))

In [ ]:
import lzma
import ijson
import geopandas as gpd
from collections import Counter

# Check that the boundary data is available
print("Boundary columns:", FINAL_BOUNDARIES.columns.tolist())
print("Boundary cities:", len(FINAL_BOUNDARIES))

In [ ]:
import lzma
import ijson
from collections import Counter
from shapely.geometry import shape
from shapely.prepared import prep
import pandas as pd

boundary_map = {
    row["city"]: prep(row.geometry)
    for _, row in FINAL_BOUNDARIES.iterrows()
}

audit_results = []

for city, path in OSM_FILES.items():
    print(f"Auditing {city}...")

    boundary = boundary_map[city]

    counts = Counter()
    total_features = 0
    matched_features = 0
    inside_features = 0
    outside_features = 0
    missing_geometry = 0
    invalid_geometry = 0
    overlaps = 0

    with lzma.open(path, "rb") as f:
        for feature in ijson.items(f, "features.item"):
            total_features += 1

            props = feature.get("properties") or {}

            matches = {
                category
                for (key, value), category in TAG_LOOKUP.items()
                if props.get(key) == value
            }

            if not matches:
                continue

            matched_features += 1

            if len(matches) > 1:
                overlaps += 1

            geom_data = feature.get("geometry")

            if not geom_data:
                missing_geometry += 1
                continue

            try:
                geom = shape(geom_data)

                if not geom.is_valid:
                    invalid_geometry += 1

                if boundary.intersects(geom):
                    inside_features += 1
                    for category in matches:
                        counts[category] += 1
                else:
                    outside_features += 1

            except Exception:
                missing_geometry += 1

    audit_results.append({
        "city": city,
        "total_osm_features": total_features,
        "tag_matches": matched_features,
        "inside_boundary": inside_features,
        "outside_boundary": outside_features,
        "missing_geometry": missing_geometry,
        "invalid_geometry": invalid_geometry,
        "overlaps": overlaps,
        **{cat: counts[cat] for cat in TAG_LOOKUP.values()}
    })

audit_df = pd.DataFrame(audit_results).fillna(0)

print("\nAUDIT SUMMARY")
display(audit_df)

In [ ]:
print("TAG_LOOKUP categories:")
print(sorted(set(TAG_LOOKUP.values())))

print("\nAUDIT DATAFRAME COLUMNS:")
print(audit_df.columns.tolist())

print("\nCATEGORY TOTALS VS INSIDE:")
category_cols = sorted(set(TAG_LOOKUP.values()))

check = audit_df[["city", "inside_boundary"]].copy()
check["category_total"] = audit_df[category_cols].sum(axis=1)
check["difference"] = check["category_total"] - check["inside_boundary"]

display(check)

In [ ]:
from collections import Counter
import lzma
import ijson

overlap_summary = {}

for city, path in OSM_FILES.items():
    counts = Counter()

    with lzma.open(path, "rb") as f:
        for feature in ijson.items(f, "features.item"):
            props = feature.get("properties") or {}

            matches = {
                category
                for (key, value), category in TAG_LOOKUP.items()
                if props.get(key) == value
            }

            if len(matches) > 1:
                counts[" + ".join(sorted(matches))] += 1

    overlap_summary[city] = counts

for city, counts in overlap_summary.items():
    if counts:
        print(city, dict(counts))

In [ ]:
import lzma
import ijson
import geopandas as gpd
import pandas as pd

from shapely.geometry import shape
from collections import Counter
from pathlib import Path

OUTPUT_DIR = PROJECT / "processed" / "recreation_facilities"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXCLUDED_ACCESS = {
    "private", "no", "residents", "members", "permit"
}

REVIEW_ACCESS = {
    "students", "children", "discouraged"
}

# Explicit rules for the overlaps found in our ten-city audit
OVERLAP_RULES = {
    frozenset(["Entertainment", "Green spaces"]): "Entertainment",
    frozenset(["Green spaces", "Sports"]): "Sports",
    frozenset(["Amusement and Play", "Green spaces"]): "Amusement and Play",
    frozenset(["Green spaces", "Outdoor activities"]): "Outdoor activities",
    frozenset(["Entertainment", "Sports"]): "Entertainment"
}

cleaned_data = {}
cleaning_summary = []

for city, path in OSM_FILES.items():

    print(f"Processing {city}...")

    boundary = FINAL_BOUNDARIES.loc[
        FINAL_BOUNDARIES["city"] == city, "geometry"
    ].iloc[0]

    records = []

    counts = Counter()
    rejected_access = Counter()
    rejected_outside = 0
    overlap_count = 0
    review_count = 0

    with lzma.open(path, "rb") as f:

        for feature in ijson.items(f, "features.item"):

            props = feature.get("properties") or {}

            matches = {
                category
                for (key, value), category in TAG_LOOKUP.items()
                if props.get(key) == value
            }

            if not matches:
                continue

            access = props.get("access")

            if access in EXCLUDED_ACCESS:
                rejected_access[access] += 1
                continue

            geom_data = feature.get("geometry")

            if not geom_data:
                continue

            geom = shape(geom_data)

            if not boundary.intersects(geom):
                rejected_outside += 1
                continue

            if len(matches) == 1:
                category = next(iter(matches))
            else:
                overlap_count += 1
                category = OVERLAP_RULES.get(frozenset(matches))

                if category is None:
                    raise ValueError(
                        f"Unresolved overlap in {city}: {matches}"
                    )

            needs_review = access in REVIEW_ACCESS

            if needs_review:
                review_count += 1

            record = props.copy()
            record["recreation_category"] = category
            record["access_review"] = needs_review

            records.append({
                **record,
                "geometry": geom
            })

            counts[category] += 1

    gdf = gpd.GeoDataFrame(
        records,
        geometry="geometry",
        crs="EPSG:4326"
    )

    cleaned_data[city] = gdf

    output_path = OUTPUT_DIR / f"{city.lower()}_recreation_cleaned.geojson"
    gdf.to_file(output_path, driver="GeoJSON")

    cleaning_summary.append({
        "city": city,
        "retained": len(gdf),
        "excluded_access": sum(rejected_access.values()),
        "outside_boundary": rejected_outside,
        "overlaps_resolved": overlap_count,
        "access_review": review_count,
        **{cat: counts[cat] for cat in sorted(set(TAG_LOOKUP.values()))}
    })

cleaning_df = pd.DataFrame(cleaning_summary)

print("\nCLEANING SUMMARY")
display(cleaning_df)

print("\nSaved files to:", OUTPUT_DIR)

In [ ]:
import pandas as pd

candidate_id_fields = [
    "id", "@id", "osm_id", "osm_type",
    "ref", "wikidata", "name",
    "recreation_category"
]

audit = []

for city, gdf in cleaned_data.items():
    row = {
        "city": city,
        "records": len(gdf),
        "geometry_types": ", ".join(
            sorted(gdf.geometry.geom_type.dropna().unique())
        )
    }

    for field in candidate_id_fields:
        if field in gdf.columns:
            row[f"{field}_present"] = int(gdf[field].notna().sum())
            row[f"{field}_unique"] = int(gdf[field].nunique())
        else:
            row[f"{field}_present"] = 0
            row[f"{field}_unique"] = 0

    audit.append(row)

identifier_audit = pd.DataFrame(audit)

display(identifier_audit)

In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd

FINAL_DIR = PROJECT / "outputs" / "final_recreation_facilities"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

final_facilities = {}
clip_summary = []

for city, gdf in cleaned_data.items():

    boundary = FINAL_BOUNDARIES.loc[
        FINAL_BOUNDARIES["city"] == city, "geometry"
    ].iloc[0]

    result = gdf.copy()
    before = len(result)

    result["geometry"] = result.geometry.apply(
        lambda geom: geom.intersection(boundary)
        if geom is not None else None
    )

    result = result[
        result.geometry.notna() &
        ~result.geometry.is_empty
    ].copy()

    result = gpd.GeoDataFrame(
        result,
        geometry="geometry",
        crs="EPSG:4326"
    )

    final_facilities[city] = result

    output_path = FINAL_DIR / f"{city.lower()}_recreation_final.geojson"
    result.to_file(output_path, driver="GeoJSON")

    clip_summary.append({
        "city": city,
        "before_clipping": before,
        "after_clipping": len(result),
        "removed": before - len(result)
    })

clip_df = pd.DataFrame(clip_summary)

display(clip_df)
print("Final datasets saved:", FINAL_DIR)

In [ ]:
import numpy as np
import pandas as pd

CATEGORIES = [
    "Green spaces",
    "Amusement and Play",
    "Sports",
    "Culture",
    "Entertainment",
    "Outdoor activities"
]

results = []

for city, gdf in final_facilities.items():

    area = FINAL_BOUNDARIES.loc[
        FINAL_BOUNDARIES["city"] == city,
        "area_km2"
    ].iloc[0]

    counts = gdf["recreation_category"].value_counts()

    category_counts = {
        cat: int(counts.get(cat, 0))
        for cat in CATEGORIES
    }

    N = sum(category_counts.values())

    proportions = [
        count / N
        for count in category_counts.values()
        if count > 0
    ]

    H = -sum(p * np.log(p) for p in proportions)
    H_normalized = H / np.log(len(CATEGORIES))

    results.append({
        "city": city,
        "total_facilities": N,
        "area_km2": area,
        "availability_per_km2": N / area,
        "shannon_H": H,
        "normalized_diversity": H_normalized,
        **category_counts
    })

recreation_metrics = pd.DataFrame(results)

display(
    recreation_metrics.round(4)
)

In [ ]:
from pathlib import Path

rasters = list(Path("/kaggle/input").rglob("*.tif"))

for p in rasters:
    print(p)

In [ ]:
POP_RASTER = Path(
    "/kaggle/input/datasets/vedantsaraswate/complete-backup-edu/"
    "geo_data/population_raw/"
    "ind_t_15_2025_CN_1km_R2025A_UA_v1.tif"
)

In [ ]:
from rasterio.mask import mask

In [ ]:
import rasterio
from rasterio.mask import mask
import geopandas as gpd
import pandas as pd
import numpy as np

demand_points = {}

with rasterio.open(POP_RASTER) as src:

    print("Raster CRS:", src.crs)
    print("Raster resolution:", src.res)
    print("Raster nodata:", src.nodata)

    for _, row in FINAL_BOUNDARIES.iterrows():

        city = row["city"]
        boundary = row.geometry

        # Extract raster cells inside the city boundary
        data, transform = mask(
            src,
            [boundary],
            crop=True,
            filled=False
        )

        arr = data[0]

        # Keep positive-population cells only
        mask_valid = (
            ~np.ma.getmaskarray(arr)
            & (arr > 0)
        )

        rows, cols = np.where(mask_valid)

        # Get cell-centre coordinates
        xs, ys = rasterio.transform.xy(
            transform,
            rows,
            cols,
            offset="center"
        )

        # Create demand-point GeoDataFrame
        points = gpd.GeoDataFrame(
            {
                "population": arr[rows, cols].data
            },
            geometry=gpd.points_from_xy(xs, ys),
            crs=src.crs
        )

        # Standardise CRS
        points = points.to_crs("EPSG:4326")

        # Retain cell centres inside the municipal boundary
        points = points[
            points.geometry.within(boundary)
        ].reset_index(drop=True)

        demand_points[city] = points

        print(f"{city}: {len(points)} demand cells")

print(
    "\nTotal demand cells:",
    sum(len(gdf) for gdf in demand_points.values())
)

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd

from scipy.spatial import cKDTree
from pathlib import Path

# CuGa1 parameters
A_THRESHOLD = 200.0
V_PARAMETER = 57708.0
SEARCH_RADIUS = 1500.0

accessibility_results = []
accessibility_points = {}

for city in OSM_FILES:

    print(f"Calculating accessibility: {city}")

    # Retrieve demand points and recreation facilities
    demand = demand_points[city].copy()
    facilities = final_facilities[city].copy()

    # Estimate a suitable local projected CRS
    boundary = FINAL_BOUNDARIES.loc[
        FINAL_BOUNDARIES["city"] == city
    ]

    local_crs = boundary.estimate_utm_crs()

    # Project demand points
    demand_projected = demand.to_crs(local_crs)

    # Convert facility geometries to representative points
    facilities = facilities[
        facilities.geometry.notna()
        & ~facilities.geometry.is_empty
    ].copy()

    facilities.geometry = facilities.geometry.representative_point()
    facilities_projected = facilities.to_crs(local_crs)

    # Extract coordinates
    demand_xy = np.column_stack([
        demand_projected.geometry.x,
        demand_projected.geometry.y
    ])

    supply_xy = np.column_stack([
        facilities_projected.geometry.x,
        facilities_projected.geometry.y
    ])

    # Build spatial index
    tree = cKDTree(supply_xy)

    # Find nearby facilities for each demand cell
    neighbours = tree.query_ball_point(
        demand_xy,
        r=SEARCH_RADIUS
    )

    cell_accessibility = np.zeros(len(demand_xy))

    for i, indices in enumerate(neighbours):

        if len(indices) == 0:
            continue

        distances = np.linalg.norm(
            supply_xy[indices] - demand_xy[i],
            axis=1
        )

        # CuGa1 impedance
        weights = np.where(
            distances <= A_THRESHOLD,
            1.0,
            np.exp(
                -((distances - A_THRESHOLD) ** 2)
                / V_PARAMETER
            )
        )

        # Every facility has opportunity weight O_j = 1
        cell_accessibility[i] = weights.sum()

    # Arithmetic mean across demand grid cells
    city_accessibility = cell_accessibility.mean()

    demand["accessibility"] = cell_accessibility
    accessibility_points[city] = demand

    accessibility_results.append({
        "city": city,
        "demand_cells": len(demand),
        "facilities": len(facilities),
        "mean_accessibility": city_accessibility,
        "median_accessibility": np.median(cell_accessibility),
        "minimum_accessibility": cell_accessibility.min(),
        "maximum_accessibility": cell_accessibility.max()
    })

accessibility_df = pd.DataFrame(accessibility_results)

print("\nACCESSIBILITY RESULTS")
display(accessibility_df.round(6))

In [ ]:
import pandas as pd
import numpy as np

# Merge the three components by city
final_recreation = (
    accessibility_df[["city", "mean_accessibility"]]
    .merge(
        recreation_metrics[
            ["city", "availability_per_km2", "normalized_diversity"]
        ],
        on="city"
    )
)

# Min-Max normalization
for col, new_col in [
    ("mean_accessibility", "A_normalized"),
    ("availability_per_km2", "V_normalized")
]:
    min_val = final_recreation[col].min()
    max_val = final_recreation[col].max()

    final_recreation[new_col] = (
        (final_recreation[col] - min_val)
        / (max_val - min_val)
    )

# Shannon diversity is already normalized
final_recreation["H_normalized"] = (
    final_recreation["normalized_diversity"]
)

# Final Recreation Index
final_recreation["Recreation_Index"] = (
    final_recreation["A_normalized"]
    * final_recreation["V_normalized"]
    + final_recreation["H_normalized"]
) / 2

# Rank cities
final_recreation = final_recreation.sort_values(
    "Recreation_Index",
    ascending=False
).reset_index(drop=True)

final_recreation.insert(
    0, "Rank", range(1, len(final_recreation) + 1)
)

display(final_recreation.round(4))

# Save results
OUTPUT_PATH = PROJECT / "outputs" / "final_recreation_index.csv"
final_recreation.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)

In [ ]:
import numpy as np
import pandas as pd

from scipy.spatial import cKDTree
from scipy.stats import spearmanr, kendalltau

# Nine parameter combinations
A_VALUES = [100, 200, 300]
V_VALUES = [28854, 57708, 115416]

SEARCH_RADIUS = 2500.0

# Preserve original baseline ranking
baseline_ranking = (
    final_recreation
    .sort_values("Recreation_Index", ascending=False)
    .set_index("city")
)

baseline_order = baseline_ranking.index.tolist()

# Fixed availability and diversity values
fixed_metrics = recreation_metrics.set_index("city")

scenario_results = []
scenario_accessibility = {}

for a in A_VALUES:
    for v in V_VALUES:

        scenario_name = f"a{a}_v{v}"

        city_access = {}

        for city in OSM_FILES:

            demand = demand_points[city]
            facilities = final_facilities[city]

            local_crs = FINAL_BOUNDARIES.loc[
                FINAL_BOUNDARIES["city"] == city
            ].estimate_utm_crs()

            demand_projected = demand.to_crs(local_crs)

            supply = facilities.copy()
            supply = supply[
                supply.geometry.notna()
                & ~supply.geometry.is_empty
            ]

            supply.geometry = supply.geometry.representative_point()
            supply_projected = supply.to_crs(local_crs)

            demand_xy = np.column_stack([
                demand_projected.geometry.x,
                demand_projected.geometry.y
            ])

            supply_xy = np.column_stack([
                supply_projected.geometry.x,
                supply_projected.geometry.y
            ])

            tree = cKDTree(supply_xy)

            neighbours = tree.query_ball_point(
                demand_xy,
                r=SEARCH_RADIUS
            )

            cell_scores = np.zeros(len(demand_xy))

            for i, indices in enumerate(neighbours):

                if not indices:
                    continue

                distances = np.linalg.norm(
                    supply_xy[indices] - demand_xy[i],
                    axis=1
                )

                weights = np.where(
                    distances <= a,
                    1.0,
                    np.exp(-((distances - a) ** 2) / v)
                )

                cell_scores[i] = weights.sum()

            city_access[city] = cell_scores.mean()

        scenario_accessibility[scenario_name] = city_access

        # Normalize accessibility within this scenario
        access_series = pd.Series(city_access)

        A_norm = (
            (access_series - access_series.min())
            / (access_series.max() - access_series.min())
        )

        # Combine with fixed availability and diversity
        V_norm = fixed_metrics.loc[
            access_series.index, "availability_per_km2"
        ]

        V_norm = (
            (V_norm - V_norm.min())
            / (V_norm.max() - V_norm.min())
        )

        H_norm = fixed_metrics.loc[
            access_series.index, "normalized_diversity"
        ]

        scores = (
            A_norm * V_norm + H_norm
        ) / 2

        scenario_results.append({
            "scenario": scenario_name,
            "a_m": a,
            "v_m2": v,
            **{
                f"{city}_score": scores[city]
                for city in scores.index
            }
        })

# Build scenario score table
scenario_df = pd.DataFrame(scenario_results)

# Compare each scenario's ranking with the original baseline
comparison = []

for _, row in scenario_df.iterrows():

    scores = {
        city: row[f"{city}_score"]
        for city in baseline_order
    }

    new_order = sorted(
        scores,
        key=scores.get,
        reverse=True
    )

    baseline_ranks = {
        city: rank
        for rank, city in enumerate(baseline_order, 1)
    }

    new_ranks = {
        city: rank
        for rank, city in enumerate(new_order, 1)
    }

    rho = spearmanr(
        list(baseline_ranks.values()),
        [new_ranks[c] for c in baseline_order]
    ).statistic

    tau = kendalltau(
        list(baseline_ranks.values()),
        [new_ranks[c] for c in baseline_order]
    ).statistic

    comparison.append({
        "scenario": row["scenario"],
        "a_m": row["a_m"],
        "v_m2": row["v_m2"],
        "spearman": rho,
        "kendall_tau": tau,
        "top_city": new_order[0],
        "bottom_city": new_order[-1]
    })

comparison_df = pd.DataFrame(comparison)

print("SCENARIO RANKING COMPARISON")
display(comparison_df.round(4))

print("\nCITY SCORES UNDER EACH SCENARIO")
display(
    scenario_df.set_index("scenario").round(4)
)

In [ ]:
import pandas as pd
from scipy.stats import spearmanr, kendalltau

# Preserve baseline ranking
baseline = final_recreation.set_index("city")

baseline_order = (
    baseline["Recreation_Index"]
    .sort_values(ascending=False)
    .index.tolist()
)

baseline_ranks = {
    city: rank
    for rank, city in enumerate(baseline_order, 1)
}

weight_results = []
weight_scores = []

for lam in [0.25, 0.50, 0.75]:

    scores = (
        lam * (
            baseline["A_normalized"]
            * baseline["V_normalized"]
        )
        + (1 - lam) * baseline["H_normalized"]
    )

    order = scores.sort_values(ascending=False).index.tolist()

    ranks = {
        city: rank
        for rank, city in enumerate(order, 1)
    }

    rho = spearmanr(
        [baseline_ranks[c] for c in baseline_order],
        [ranks[c] for c in baseline_order]
    ).statistic

    tau = kendalltau(
        [baseline_ranks[c] for c in baseline_order],
        [ranks[c] for c in baseline_order]
    ).statistic

    weight_results.append({
        "lambda": lam,
        "spearman": rho,
        "kendall_tau": tau,
        "top_city": order[0],
        "bottom_city": order[-1]
    })

    for city in baseline_order:
        weight_scores.append({
            "lambda": lam,
            "city": city,
            "score": scores[city],
            "rank": ranks[city]
        })

weight_comparison = pd.DataFrame(weight_results)
weight_score_table = pd.DataFrame(weight_scores)

print("AGGREGATION SENSITIVITY")
display(weight_comparison.round(4))

print("CITY RANKS UNDER EACH WEIGHT")
display(
    weight_score_table.pivot(
        index="city",
        columns="lambda",
        values="rank"
    )
)

In [ ]:
import numpy as np
import pandas as pd

from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix
from scipy.stats import spearmanr, kendalltau, rankdata

# Simulation settings
REMOVAL_LEVELS = [0.05, 0.10, 0.15]
REPETITIONS = 100
RANDOM_SEED = 42

A_THRESHOLD = 200.0
V_PARAMETER = 57708.0
SEARCH_RADIUS = 1500.0

rng = np.random.default_rng(RANDOM_SEED)

CATEGORIES = [
    "Green spaces",
    "Amusement and Play",
    "Sports",
    "Culture",
"Entertainment",
    "Outdoor activities"
]

cities = list(OSM_FILES.keys())

# Original baseline ranks
baseline_scores = (
    final_recreation
    .set_index("city")["Recreation_Index"]
    .reindex(cities)
)

baseline_ranks = rankdata(
    -baseline_scores.to_numpy(),
    method="average"
)

# Fixed city areas
city_areas = (
    FINAL_BOUNDARIES
    .set_index("city")["area_km2"]
    .to_dict()
)

# Precompute sparse accessibility matrices
access_matrices = {}
facility_categories = {}
facility_totals = {}

for city in cities:

    print("Preparing:", city)

    demand = demand_points[city]
    facilities = final_facilities[city].copy()

    local_crs = FINAL_BOUNDARIES.loc[
        FINAL_BOUNDARIES["city"] == city
    ].estimate_utm_crs()

    demand_projected = demand.to_crs(local_crs)

    facilities = facilities[
        facilities.geometry.notna()
        & ~facilities.geometry.is_emptis_empty
    ].copy()

    facilities.geometry = (
        facilities.geometry.representative_point()
    )

    facilities_projected = facilities.to_crs(local_crs)

    demand_xy = np.column_stack([
        demand_projected.geometry.x,
        demand_projected.geometry.y
    ])

    supply_xy = np.column_stack([
        facilities_projected.geometry.x,
        facilities_projected.geometry.y
    ])

    tree = cKDTree(supply_xy)

    neighbours = tree.query_ball_point(
demand_xy,
        r=SEARCH_RADIUS
    )

    row_indices = []
    col_indices = []
    weights = []

    for i, indices in enumerate(neighbours):

        if not indices:
            continue

        distances = np.linalg.norm(
            supply_xy[indices] - demand_xy[i],
            axis=1
        )

        impedance = np.where(
            distances <= A_THRESHOLD,
            1.0,
            np.exp(
                -((distances - A_THRESHOLD) ** 2)
                / V_PARAMETER
            )
        )

        row_indices.extend([i] * len(indices))
        col_indices.extend(indices)
        weights.extend(impedance)

    W = csr_matrix(
        (weights, (row_indices, col_indices)),
        shape=(len(demand), len(facilities))
    )

    access_matrices[city] = W

    facility_categories[city] = (
        facilities["recreation_category"].to_numpy()
    )

    facility_totals[city] = len(facilities)

print("\nDistance matrices prepared for all cities.")

# Run random facility removal simulations
simulation_records = []

for removal_rate in REMOVAL_LEVELS:

    print(f"\nSimulating {removal_rate:.0%} facility removal...")

    for repetition in range(REPETITIONS):

        city_A = {}
        city_V = {}
        city_H = {}

        for city in cities:

            W = access_matrices[city]
            categories = facility_categories[city]

            n = len(categories)

            keep = rng.random(n) >= removal_rate

            # Accessibility: arithmetic mean across demand cells
            cell_access = np.asarray(
                W @ keep.astype(float)
            ).ravel()

            city_A[city] = cell_access.mean()

            # Availability
            kept_categories = categories[keep]
            N = len(kept_categories)

            city_V[city] = N / city_areas[city]

            # Shannon diversity
            counts = np.array([
                np.sum(kept_categories == category)
                for category in CATEGORIES
            ])

            proportions = counts[counts > 0] / N

            H = -np.sum(
                proportions * np.log(proportions)
            )

            city_H[city] = H / np.log(len(CATEGORIES))

        # Normalize components across cities
        A_values = np.array([city_A[c] for c in cities])
        V_values = np.array([city_V[c] for c in cities])
        H_values = np.array([city_H[c] for c in cities])

        A_norm = (
            (A_values - A_values.min())
            / (A_values.max() - A_values.min())
        )

        V_norm = (
            (V_values - V_values.min())
            / (V_values.max() - V_values.min())
        )

        scores = (
            A_norm * V_norm + H_values
        ) / 2

        ranks = rankdata(-scores, method="average")

        rho = spearmanr(
            baseline_ranks, ranks
        ).statistic

        tau = kendalltau(
            baseline_ranks, ranks
        ).statistic

        for i, city in enumerate(cities):

            simulation_records.append({
                "removal_rate": removal_rate,
                "repetition": repetition,
                "city": city,
                "score": scores[i],
                "rank": ranks[i],
                "spearman": rho,
                "kendall_tau": tau
            })

simulation_df = pd.DataFrame(simulation_records)

# Summary by removal level
correlation_summary = (
    simulation_df.groupby("removal_rate")
    .agg(
        mean_spearman=("spearman", "mean"),
        sd_spearman=("spearman", "std"),
        mean_kendall=("kendall_tau", "mean"),
        sd_kendall=("kendall_tau", "std")
    )
    .reset_index()
)

# Summary of city rank stability
rank_summary = (
    simulation_df.groupby(["removal_rate", "city"])
    .agg(
        mean_rank=("rank", "mean"),
        sd_rank=("rank", "std"),
        best_rank=("rank", "min"),
        worst_rank=("rank", "max"),
        top_three_frequency=(
            "rank",
            lambda x: (x <= 3).mean()
        )
    )
    .reset_index()
)

print("\nRANK CORRELATION SUMMARY")
display(correlation_summary.round(4))

print("\nCITY RANK STABILITY")
display(rank_summary.round(3))

In [ ]:
# Show relevant DataFrames and dictionaries currently in memory
for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        print(f"\nDATAFRAME: {name}")
        print("Shape:", obj.shape)
        print("Columns:", list(obj.columns))

    elif isinstance(obj, dict) and any(
        isinstance(v, (pd.DataFrame, gpd.GeoDataFrame))
        for v in obj.values()
    ):
        print(f"\nDICTIONARY: {name}")
        print("Keys:", list(obj.keys())[:15])

In [ ]:
import pandas as pd
import numpy as np

# Baseline scenario: a=200m, v=57708m²
base = scenario_df[
    (scenario_df["a_m"] == 200) &
    (scenario_df["v_m2"] == 57708)
].iloc[0]

cities = [
    "Ahmedabad", "Mumbai", "Pune", "Jaipur", "Indore",
    "Hyderabad", "Chennai", "Lucknow", "Bhubaneswar", "Kolkata"
]

rows = []

for city in cities:
    col = f"{city}_score"
    baseline_score = base[col]

    values = scenario_df[col]

    rows.append({
        "City": city,
        "Baseline accessibility": baseline_score,
        "Minimum accessibility": values.min(),
        "Maximum accessibility": values.max(),
        "Absolute range": values.max() - values.min(),
        "Maximum change (%)": (
            (values - baseline_score).abs().max()
            / abs(baseline_score) * 100
            if baseline_score != 0 else np.nan
        )
    })

variation_df = pd.DataFrame(rows)

display(
    variation_df.sort_values(
        "Maximum change (%)", ascending=False
    ).round(5)
)

print(
    "\nLargest absolute percentage change:",
    round(variation_df["Maximum change (%)"].max(), 3), "%"
)

In [ ]:
# Inspect all nine raw accessibility values for each city

cols = [
    "scenario", "a_m", "v_m2",
    "Ahmedabad_score", "Mumbai_score",
    "Pune_score", "Jaipur_score", "Indore_score",
    "Hyderabad_score", "Chennai_score",
    "Lucknow_score", "Bhubaneswar_score",
    "Kolkata_score"
]

display(scenario_df[cols].round(8).to_string(index=False))

In [ ]:
# Check whether the accessibility values are exactly identical
# and inspect their variation at full precision.

for city in cities:
    col = f"{city}_score"
    values = scenario_df[col].to_numpy()

    print(
        f"{city:15s} "
        f"unique={len(np.unique(values))} | "
        f"range={np.ptp(values):.12f} | "
        f"std={np.std(values):.12f}"
    )

In [ ]:
# Inspect the actual sensitivity calculation code/results
# and identify stored distance-related objects.

for name, obj in globals().copy().items():
    if any(term in name.lower() for term in [
        "dist", "weight", "access", "sensitivity", "scenario"
    ]):
        if isinstance(obj, np.ndarray):
            print(name, obj.shape, obj.dtype)
        elif isinstance(obj, (dict, pd.DataFrame)):
            print(name, type(obj).__name__, len(obj))

In [ ]:
print("SCENARIO ACCESSIBILITY")
for key, value in scenario_accessibility.items():
    print(key, type(value), 
          value.shape if hasattr(value, "shape") else len(value))

print("\nACCESS MATRICES")
for city, value in access_matrices.items():
    print(city, type(value),
          value.shape if hasattr(value, "shape") else len(value))

In [ ]:
print("\nAHMEDABAD MATRIX")
print(access_matrices["Ahmedabad"])

print("\nAHMEDABAD SCENARIO RESULTS")
print(scenario_accessibility)

In [ ]:
import pandas as pd
import numpy as np

rows = []

for scenario, city_values in scenario_accessibility.items():
    for city, score in city_values.items():
        rows.append({
            "scenario": scenario,
            "city": city,
            "accessibility": float(score)
        })

raw_df = pd.DataFrame(rows)

baseline_raw = raw_df[
    raw_df["scenario"] == "a200_v57708"
][["city", "accessibility"]].rename(
    columns={"accessibility": "baseline"}
)

raw_df = raw_df.merge(baseline_raw, on="city")

raw_df["change_pct"] = (
    (raw_df["accessibility"] - raw_df["baseline"])
    / raw_df["baseline"] * 100
)

summary = raw_df.groupby("city").agg(
    baseline=("baseline", "first"),
    minimum=("accessibility", "min"),
    maximum=("accessibility", "max"),
    absolute_range=("accessibility", lambda x: x.max() - x.min()),
    maximum_change_pct=("change_pct", lambda x: x.abs().max())
).sort_values("maximum_change_pct", ascending=False)

display(summary.round(4))


In [ ]:
# Calculate Spearman and Kendall correlations
# for RAW accessibility across all nine scenarios.

from scipy.stats import spearmanr, kendalltau
from itertools import combinations

wide = raw_df.pivot(
    index="city",
    columns="scenario",
    values="accessibility"
)

results = []

for s1, s2 in combinations(wide.columns, 2):
    rho, _ = spearmanr(wide[s1], wide[s2])
    tau, _ = kendalltau(wide[s1], wide[s2])

    results.append({
        "Scenario 1": s1,
        "Scenario 2": s2,
        "Spearman": rho,
        "Kendall": tau
    })

correlation_df = pd.DataFrame(results)

display(correlation_df.round(4))

print("\nMinimum Spearman:", correlation_df["Spearman"].min())
print("Minimum Kendall:", correlation_df["Kendall"].min())
print("Mean Spearman:", correlation_df["Spearman"].mean())
print("Mean Kendall:", correlation_df["Kendall"].mean())

In [ ]:
required = [
    "access_matrices",
    "final_facilities",
    "accessibility_df",
    "recreation_metrics",
    "fixed_metrics"
]

for name in required:
    print(name, ":", name in globals())